# Module 10: Capstone — Putting It All Together

**ACL2 for Computer Science — University-Level Tutorial Series**

## Learning Objectives

In this final module you will:

1. Review the key concepts from all previous modules
2. Complete three capstone projects that integrate multiple topics
3. Explore the broader ACL2 ecosystem and future directions
4. Have a clear path for continued learning

## 10.1 Course Review

Let's summarize the journey through formal verification with ACL2:

| Module | Topic | Key Concepts |
|--------|-------|---------------|
| 1 | Introduction to ACL2 | S-expressions, REPL, basic evaluation |
| 2 | Propositional Logic | Boolean connectives, truth tables, tautologies |
| 3 | Recursive Functions & Induction | `defun`, termination, `defthm`, inductive proofs |
| 4 | Lists and Trees | `cons`, `car`, `cdr`, `append`, list induction |
| 5 | Sorting and Searching | Insertion sort, merge sort, binary search, verified algorithms |
| 6 | Machine Models | Stack machines, compilers, interpreters, ISA verification |
| 7 | Hardware Verification | Boolean gates, adders, bit vectors, FM9001 |
| 8 | SAT Solving | CNF, DPLL, unit propagation, BDDs |
| 9 | Cryptography | Modular arithmetic, RSA, hash functions, verified crypto |

### The Verification Methodology

Throughout this course, we have followed a consistent methodology:

1. **Define** — Write a precise ACL2 function that models the system
2. **Specify** — State the desired property as a theorem
3. **Prove** — Use ACL2's automated reasoning to verify the property

This methodology scales from simple arithmetic facts to industrial processor verification.

## 10.2 Capstone Project 1: Verified Calculator

Our first capstone ties together **recursion, induction, and machine models** (Modules 3, 4, and 6).

We will:
1. Define a simple expression language
2. Write an interpreter for it
3. Write a compiler targeting a stack machine
4. **Prove** the compiler correct: running compiled code gives the same result as interpretation

### The Expression Language

Our expressions are:
- `(NUM n)` — a numeric literal
- `(ADD e1 e2)` — addition of two sub-expressions
- `(MUL e1 e2)` — multiplication of two sub-expressions

For example, `(ADD (NUM 3) (MUL (NUM 2) (NUM 5)))` represents $3 + (2 \times 5) = 13$.

In [ ]:
; Recognizer for well-formed expressions
(defun exprp (e)
  (cond ((and (true-listp e)
              (equal (len e) 2)
              (equal (first e) 'NUM)
              (natp (second e)))
         t)
        ((and (true-listp e)
              (equal (len e) 3)
              (or (equal (first e) 'ADD)
                  (equal (first e) 'MUL))
              (exprp (second e))
              (exprp (third e)))
         t)
        (t nil)))

In [ ]:
; Test: well-formed expressions
(list (exprp '(NUM 5))                           ; t
      (exprp '(ADD (NUM 3) (NUM 4)))             ; t
      (exprp '(MUL (NUM 2) (ADD (NUM 1) (NUM 3)))) ; t
      (exprp '(SUB (NUM 1) (NUM 2))))            ; nil (SUB not supported)

### The Interpreter

The interpreter directly computes the value of an expression by recursive evaluation:

In [ ]:
; Expression interpreter
(defun eval-expr (e)
  (cond ((equal (first e) 'NUM)
         (second e))
        ((equal (first e) 'ADD)
         (+ (eval-expr (second e))
            (eval-expr (third e))))
        ((equal (first e) 'MUL)
         (* (eval-expr (second e))
            (eval-expr (third e))))
        (t 0)))

In [ ]:
; Test the interpreter
(list (eval-expr '(NUM 5))                              ; 5
      (eval-expr '(ADD (NUM 3) (NUM 4)))                ; 7
      (eval-expr '(MUL (NUM 2) (ADD (NUM 1) (NUM 3))))  ; 8
      (eval-expr '(ADD (MUL (NUM 3) (NUM 4)) (NUM 1)))) ; 13

### The Stack Machine

Our target machine is a simple stack machine (as in Module 6) with three instructions:
- `(PUSH n)` — push a number onto the stack
- `(IADD)` — pop two values, push their sum
- `(IMUL)` — pop two values, push their product

In [ ]:
; Execute a single stack machine instruction
(defun step-instr (instr stack)
  (cond ((equal (first instr) 'PUSH)
         (cons (second instr) stack))
        ((equal (first instr) 'IADD)
         (cons (+ (first stack) (second stack))
               (cddr stack)))
        ((equal (first instr) 'IMUL)
         (cons (* (first stack) (second stack))
               (cddr stack)))
        (t stack)))

In [ ]:
; Execute a program (list of instructions) on a stack
(defun run-program (program stack)
  (if (endp program)
      stack
    (run-program (cdr program)
                 (step-instr (car program) stack))))

In [ ]:
; Test: compute 3 + 4 on the stack machine
(run-program '((PUSH 3) (PUSH 4) (IADD)) nil)
; → (7)

### The Compiler

The compiler translates expressions into stack machine code:

In [ ]:
; Compiler: expression → stack machine program
(defun compile-expr (e)
  (cond ((equal (first e) 'NUM)
         (list (list 'PUSH (second e))))
        ((equal (first e) 'ADD)
         (append (compile-expr (second e))
                 (compile-expr (third e))
                 (list '(IADD))))
        ((equal (first e) 'MUL)
         (append (compile-expr (second e))
                 (compile-expr (third e))
                 (list '(IMUL))))
        (t nil)))

In [ ]:
; Test the compiler
(compile-expr '(ADD (NUM 3) (MUL (NUM 2) (NUM 5))))
; → ((PUSH 3) (PUSH 2) (PUSH 5) (IMUL) (IADD))

In [ ]:
; Run compiled code and check against interpreter
(let* ((expr '(ADD (NUM 3) (MUL (NUM 2) (NUM 5)))))
  (list :interpreted (eval-expr expr)
        :compiled-result (car (run-program (compile-expr expr) nil))
        :match (equal (eval-expr expr)
                      (car (run-program (compile-expr expr) nil)))))

### Proving Compiler Correctness

The key theorem: **running compiled code on an empty stack produces the same result as interpreting the expression.**

We first prove a more general lemma about running compiled code on *any* stack:

In [ ]:
; Lemma: compiled code pushes the expression's value onto the stack.
; (Running compiled code on stack s leaves (eval-expr e) on top of s.)
(defthm compile-expr-correct-lemma
  (equal (run-program (compile-expr e) stack)
         (cons (eval-expr e) stack)))

In [ ]:
; Main theorem: compiler correctness
(defthm compile-expr-correct
  (equal (car (run-program (compile-expr e) nil))
         (eval-expr e)))

**This is a powerful result.** It says that for *any* expression — no matter how deeply nested — the compiler produces code that computes the correct answer. The proof is by structural induction on the expression tree, which ACL2 handles automatically.

## 10.3 Capstone Project 2: Verified Key-Value Store

This project ties together **lists, association lists, and algebraic properties** (Modules 3 and 4).

We build a key-value store supporting:
- `kv-put` — insert or update a key-value pair
- `kv-get` — retrieve a value by key
- `kv-delete` — remove a key

We prove the fundamental **CRUD properties** that any correct key-value store must satisfy.

In [ ]:
; An empty store
(defun kv-empty () nil)

In [ ]:
; Put: insert or update a key-value pair.
; We use a simple association list representation.
(defun kv-put (key value store)
  (cons (cons key value)
        (kv-delete key store)))

In [ ]:
; Get: retrieve a value by key. Returns nil if not found.
(defun kv-get (key store)
  (cond ((endp store) nil)
        ((equal key (car (car store)))
         (cdr (car store)))
        (t (kv-get key (cdr store)))))

In [ ]:
; Delete: remove a key from the store.
(defun kv-delete (key store)
  (cond ((endp store) nil)
        ((equal key (car (car store)))
         (kv-delete key (cdr store)))
        (t (cons (car store)
                 (kv-delete key (cdr store))))))

In [ ]:
; Test the key-value store
(let* ((s0 (kv-empty))
       (s1 (kv-put 'name 'alice s0))
       (s2 (kv-put 'age 30 s1))
       (s3 (kv-put 'name 'bob s2)))  ; update name
  (list :name (kv-get 'name s3)      ; bob
        :age  (kv-get 'age s3)       ; 30
        :missing (kv-get 'email s3)  ; nil
        ))

### The CRUD Properties

These properties characterize the *algebraic specification* of a key-value store:

In [ ]:
; Property 1: Get after Put (same key) returns the value
(defthm kv-get-put-same
  (equal (kv-get k (kv-put k v store))
         v))

In [ ]:
; Property 2: Get after Put (different key) is unchanged
(defthm kv-get-put-different
  (implies (not (equal k1 k2))
           (equal (kv-get k1 (kv-put k2 v store))
                  (kv-get k1 store))))

In [ ]:
; Property 3: Get after Delete returns nil
(defthm kv-get-delete
  (equal (kv-get k (kv-delete k store))
         nil))

In [ ]:
; Property 4: Delete is idempotent
(defthm kv-delete-idempotent
  (equal (kv-delete k (kv-delete k store))
         (kv-delete k store)))

In [ ]:
; Property 5: Put overwrites previous values
(defthm kv-put-overwrite
  (equal (kv-get k (kv-put k v2 (kv-put k v1 store)))
         v2))

In [ ]:
; Property 6: Delete after Put (same key) removes it
(defthm kv-delete-after-put-same
  (equal (kv-get k (kv-delete k (kv-put k v store)))
         nil))

These six properties form a complete algebraic specification of a correct key-value store. Any implementation satisfying these properties behaves correctly regardless of its internal representation.

## 10.4 Capstone Project 3: Verified Sorting Pipeline

This project ties together **sorting, list operations, and end-to-end verification** (Modules 4 and 5).

We build a pipeline: **input list → sort → deduplicate → output**, and prove:
1. The output is sorted
2. The output contains no duplicates
3. Every element in the output appears in the input

In [ ]:
; Insertion sort (from Module 5)
(defun insert-sorted (x lst)
  (cond ((endp lst) (list x))
        ((<= x (car lst)) (cons x lst))
        (t (cons (car lst)
                 (insert-sorted x (cdr lst))))))

(defun isort (lst)
  (if (endp lst)
      nil
    (insert-sorted (car lst)
                   (isort (cdr lst)))))

In [ ]:
; Remove consecutive duplicates from a sorted list
(defun dedup (lst)
  (cond ((endp lst) nil)
        ((endp (cdr lst)) (list (car lst)))
        ((equal (car lst) (cadr lst))
         (dedup (cdr lst)))
        (t (cons (car lst)
                 (dedup (cdr lst))))))

In [ ]:
; The complete pipeline
(defun sort-dedup-pipeline (input)
  (dedup (isort input)))

In [ ]:
; Test the pipeline
(sort-dedup-pipeline '(3 1 4 1 5 9 2 6 5 3 5))

### Proving Pipeline Properties

In [ ]:
; Helper: check if a list is sorted
(defun sortedp (lst)
  (cond ((endp lst) t)
        ((endp (cdr lst)) t)
        ((<= (car lst) (cadr lst))
         (sortedp (cdr lst)))
        (t nil)))

In [ ]:
; Helper: check if a list has no duplicates
(defun no-dupsp (lst)
  (cond ((endp lst) t)
        ((member-equal (car lst) (cdr lst)) nil)
        (t (no-dupsp (cdr lst)))))

In [ ]:
; Property 1: isort produces a sorted list
(defthm isort-sorts
  (sortedp (isort lst)))

In [ ]:
; Lemma: dedup preserves sortedness
(defthm dedup-preserves-sorted
  (implies (sortedp lst)
           (sortedp (dedup lst))))

In [ ]:
; Property 2: The pipeline output is sorted
(defthm pipeline-sorted
  (sortedp (sort-dedup-pipeline input)))

In [ ]:
; Property 3: dedup removes all duplicates from a sorted list
(defthm dedup-no-dups
  (implies (sortedp lst)
           (no-dupsp (dedup lst))))

In [ ]:
; Property 4: The pipeline output has no duplicates
(defthm pipeline-no-dups
  (no-dupsp (sort-dedup-pipeline input)))

In [ ]:
; Property 5: Every element in the output was in the input
(defthm pipeline-subset
  (implies (member-equal x (sort-dedup-pipeline input))
           (member-equal x input)))

We have proved **end-to-end** that our pipeline:
- Produces sorted output ✓
- Contains no duplicates ✓  
- Only contains elements from the input ✓

This is the power of formal verification: we didn't just test a few cases — we proved these properties hold for **all possible inputs**.

## 10.5 The ACL2 Ecosystem

ACL2 is much more than a theorem prover — it's a thriving ecosystem of tools, libraries, and research projects.

### Community Books

The ACL2 **Community Books** are a vast, collaboratively maintained library of formal developments:

- **14,500+ Lisp files** spanning dozens of research areas
- Contributed by researchers and practitioners worldwide
- Curated with regression testing to ensure compatibility

Key directories include:

| Directory | Content |
|-----------|----------|
| `books/std/` | Standard libraries (lists, alists, strings, etc.) |
| `books/arithmetic-5/` | Arithmetic reasoning |
| `books/projects/` | Major verification projects (FM9001, SHA-2, x86, etc.) |
| `books/centaur/` | Hardware verification tools (from industry) |
| `books/kestrel/` | Kestrel Institute contributions (crypto, Java, APT) |
| `books/coi/` | Computational Object Infrastructure |
| `books/clause-processors/` | SAT/SMT integration |

### Key Research Areas

**Processor Verification**
- FM9001 (complete processor)
- x86 ISA model (`books/projects/x86isa/`)
- JVM model (`books/models/jvm/`)

**Cryptography**
- SHA-256, AES, elliptic curves
- Verified crypto implementations

**Compiler Verification**
- Expression compilers (as in this module)
- APT: Automated Program Transformations

**Operating Systems**
- File system verification
- Memory management proofs

**Security**
- Information flow analysis
- Access control verification

### ACL2 Variants and Related Systems

| System | Description |
|--------|-------------|
| **ACL2** | The main theorem prover (Kaufmann & Moore) |
| **ACL2s** | ACL2 Sedan — enhanced for teaching, with better defaults and IDE |
| **ACL2(r)** | Extension for reasoning about real numbers |
| **Milawa** | A simpler theorem prover verified in ACL2 (self-verifying!) |
| **APT** | Automated Program Transformations toolkit |
| **GL** | Bit-level symbolic execution for hardware verification |

## 10.6 Further Reading

### Textbooks

- **"Computer-Aided Reasoning: An Approach"** by Kaufmann, Manolios, and Moore — The definitive ACL2 textbook
- **"Computer-Aided Reasoning: ACL2 Case Studies"** by Kaufmann, Manolios, and Moore — Real-world verification case studies
- **"A Computational Logic Handbook"** by Boyer and Moore — Foundations of the ACL2 approach
- **"Structured Programming"** by Dahl, Dijkstra, and Hoare — Background on program correctness

### Key Papers

- Hunt, W.A. "FM8501: A Verified Microprocessor" — Landmark processor verification
- Kaufmann, M. and Moore, J S. "An Industrial Strength Theorem Prover for a Logic Based on Common Lisp" — ACL2 design paper
- Russinoff, D. "A Mechanically Checked Proof of IEEE Compliance of the Floating Point Multiplication, Division, and Square Root Algorithms of the AMD-K7 Processor" — Industrial application

### Online Resources

- [ACL2 Home Page](https://www.cs.utexas.edu/~moore/acl2/) — Official documentation and downloads
- [ACL2 Community Books](https://github.com/acl2/acl2) — Source code and community contributions
- [ACL2s](https://acl2s.ccs.neu.edu/) — The teaching-oriented ACL2 system

## Congratulations!

You have completed the **ACL2 for Computer Science** tutorial series. You now have:

- **Foundational knowledge** of theorem proving and formal verification
- **Practical experience** writing ACL2 definitions and proofs
- **Exposure to real applications** from hardware to cryptography
- **Capstone projects** demonstrating end-to-end verification

Formal verification is an increasingly important skill in computer science. The techniques you've learned here — modeling systems precisely, stating properties formally, and proving them mechanically — apply far beyond ACL2 to any domain where correctness matters.

> *"Beware of bugs in the above code; I have only proved it correct, not tried it."*
> — Donald Knuth

---

**Navigation:**
[< Module 9 — Cryptography and Security](09_crypto_security.ipynb) | **Course Complete**